# Lesson 3: Enable Logging

### Import all needed packages

In [1]:
import boto3
import json
import os

bedrock = boto3.client('bedrock', region_name="us-west-2")

In [2]:
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [3]:
from helpers.CloudWatchHelper import CloudWatch_Helper
cloudwatch = CloudWatch_Helper()

In [4]:
log_group_name = '/my/amazon/bedrock/logs'

In [5]:
cloudwatch.create_log_group(log_group_name)

Log group '/my/amazon/bedrock/logs' already exists.


In [6]:
loggingConfig = {
    'cloudWatchConfig': {
        'logGroupName': log_group_name,
        'roleArn': os.environ['LOGGINGROLEARN'],
        'largeDataDeliveryS3Config': {
            'bucketName': os.getenv('BucketName'),
            'keyPrefix': 'amazon_bedrock_large_data_delivery',
        }
    },
    's3Config': {
        'bucketName': os.getenv('BucketName'),
        'keyPrefix': 'amazon_bedrock_logs',
    },
    'textDataDeliveryEnabled': True,
}

In [7]:
bedrock.put_model_invocation_logging_configuration(loggingConfig=loggingConfig)

{'ResponseMetadata': {'RequestId': '9ce3abe6-c5d5-47d7-9d77-5902bdb66698',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sun, 10 Aug 2025 23:52:50 GMT',
   'content-type': 'application/json',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-amzn-requestid': '9ce3abe6-c5d5-47d7-9d77-5902bdb66698'},
  'RetryAttempts': 0}}

In [8]:
bedrock.get_model_invocation_logging_configuration()

{'ResponseMetadata': {'RequestId': '9a03ac7a-1afa-4c52-96c7-7d2bcc4e66a1',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sun, 10 Aug 2025 23:52:52 GMT',
   'content-type': 'application/json',
   'content-length': '483',
   'connection': 'keep-alive',
   'x-amzn-requestid': '9a03ac7a-1afa-4c52-96c7-7d2bcc4e66a1'},
  'RetryAttempts': 0},
 'loggingConfig': {'cloudWatchConfig': {'logGroupName': '/my/amazon/bedrock/logs',
   'roleArn': 'arn:aws:iam::092413168457:role/CloudWatchES-Course',
   'largeDataDeliveryS3Config': {'bucketName': 'elephantscale-public-temp',
    'keyPrefix': 'amazon_bedrock_large_data_delivery'}},
  's3Config': {'bucketName': 'elephantscale-public-temp',
   'keyPrefix': 'amazon_bedrock_logs'},
  'textDataDeliveryEnabled': True,
  'imageDataDeliveryEnabled': True,
  'embeddingDataDeliveryEnabled': True}}

In [9]:
bedrock_runtime = boto3.client('bedrock-runtime', region_name="us-west-2")

In [10]:
prompt = "Write an article about the fictional planet Foobar."

kwargs = {
    "modelId": "amazon.titan-text-express-v1",
    "contentType": "application/json",
    "accept": "*/*",
    "body": json.dumps(
        {
            "inputText": prompt,
            "textGenerationConfig": {
                "maxTokenCount": 512,
                "temperature": 0.7,
                "topP": 0.9
            }
        }
    )
}

response = bedrock_runtime.invoke_model(**kwargs)
response_body = json.loads(response.get('body').read())

generation = response_body['results'][0]['outputText']

print(generation)


Foobar is a fictional planet featured in the popular science fiction novel "Dune" by Frank Herbert. The planet is located in the far future and is the home of the Fremen, a group of humans who have adapted to the harsh desert environment of Arrakis, also known as Dune.

Foobar is a harsh, desert planet with a thin atmosphere and high levels of radiation. The Fremen have adapted to these conditions by wearing protective clothing and helmets, and by living in underground shelters. The planet is ruled by a feudal system, with the Padishah Emperor being the ultimate ruler.

The Fremen are led by a group of warrior-priests known as the Fremen Masters. The Masters are responsible for the training of the Fremen and for the protection of their homeworld. They have a deep understanding of the desert environment and its dangers, and they use this knowledge to their advantage in battle.

The Fremen are a fierce and independent people who have a strong sense of loyalty to their Master and to thei

In [11]:
cloudwatch.print_recent_logs(log_group_name)

{
    "timestamp": "2025-08-10T23:49:22Z",
    "accountId": "092413168457",
    "identity": {
        "arn": "arn:aws:iam::092413168457:user/rushi"
    },
    "region": "us-west-2",
    "requestId": "43d21369-080e-49e0-8f81-ac5694bc8919",
    "operation": "InvokeModel",
    "modelId": "amazon.titan-text-express-v1",
    "input": {
        "inputContentType": "application/json",
        "inputBodyJson": {
            "inputText": "I need to summarize a conversation. The transcript of the \nconversation is between the <data> XML like tags.\n\n<data>\n\nspk_0: Hi, is this the Crystal Heights Hotel in Singapore? \nspk_1: Yes, it is. Good afternoon. How may I assist you today? \nspk_0: Fantastic. Good afternoon. I was looking to book a room for my 10th wedding anniversary. I've heard your hotel offers exceptional views and services. Could you tell me more? \nspk_1: Absolutely, Alex, and congratulations on your upcoming anniversary. That's a significant milestone, and we'd be honored to make

To review the logs within the AWS console, please use the following link to reference the steps outlined in the video: